In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('3-24-DCN-API.csv', index_col=0)

C:\Users\wieke\AppData\Local\Temp\ipykernel_21308\3857647716.py:1: DtypeWarning: Columns (12,13,19,20,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('3-24-DCN-API.csv', index_col=0)


In [3]:
len(df)

563232

In [ ]:
pd.set_option("display.max_columns", 55)


In [ ]:
def counts(col1, col2, col3, df):
    oa_count_s = df.groupby([col1])[col2].value_counts().unstack(col2).reset_index()
    oa_count_s = oa_count_s.fillna(0)
    
    oa_count_s[col3] = oa_count_s[True] / (oa_count_s[False] + oa_count_s[True])
    oa_count_s['Total'] = oa_count_s[True] + oa_count_s[False]
    oa_count_s
    return oa_count_s.sort_values(by=[col3], ascending=False)
    



In [ ]:
def multiple_counts(col1, col2, col3, col4, df):
    oa_count_s = df.groupby([col1, col2])[col3].value_counts().unstack(col3).reset_index()
    oa_count_s = oa_count_s.fillna(0)
    
    oa_count_s[col4] = oa_count_s[True] / (oa_count_s[False] + oa_count_s[True])
    oa_count_s['Total'] = oa_count_s[True] + oa_count_s[False]

    
    return oa_count_s


In [ ]:
def oa_status(df, col1, col2, sorted):
    oa_status_df = df.groupby([col1])[col2].value_counts().unstack(col2).reset_index()
    oa_status_df = oa_status_df.fillna(0)
    oa_status_df['total'] = oa_status_df['bronze'] + oa_status_df['closed'] + oa_status_df['gold'] + oa_status_df['green'] + oa_status_df['hybrid']
    oa_status_df['hybrid %'] = oa_status_df['hybrid'] / oa_status_df['total']
    oa_status_df['green %'] = oa_status_df['green'] / oa_status_df['total']
    oa_status_df['gold %'] = oa_status_df['gold'] / oa_status_df['total']
    oa_status_df['closed %'] = oa_status_df['closed'] / oa_status_df['total']
    oa_status_df['bronze %'] = oa_status_df['bronze'] / oa_status_df['total']
    dct = {'hybrid %': 'hybrid', 'green %': 'green', 'gold %': 'gold', 'closed %': 'closed', 'bronze %': 'bronze'}
    oa_status_df['Most used OA type'] = oa_status_df[['hybrid %', 'green %', 'gold %', 'closed %', 'bronze %']].idxmax(axis=1).map(dct)

    #oa_status_df = oa_status_df.sort_values(by=[sort], ascending=False)
    oa_status_df = oa_status_df.sort_values(by=[sorted], ascending=False).reset_index(drop=True)

    return oa_status_df

In [ ]:
def RadsCounts(col1, col2, df, sort):
    #oa_count_s = df.groupby([[col1, col2]])[col3].value_counts().unstack(col3).reset_index()
    oa_count_s = df.groupby([col1])[col2].value_counts().unstack(col2).reset_index()
    #oa_count_s['total'] = df.groupby['publisher']['DOI'].size()
    
    
    #oa_count_s = oa_count_s.fillna(0)
    
    return oa_count_s.sort_values(by=[sort], ascending=False).reset_index(drop=True)

In [ ]:
def RadsMultipleCounts(col1, col2, col3, df, sort):
    #oa_count_s = df.groupby([[col1, col2]])[col3].value_counts().unstack(col3).reset_index()
    oa_count_s = df.groupby([col1, col2])[col3].value_counts().unstack(col3).reset_index()
    #oa_count_s['total'] = df.groupby['publisher']['DOI'].size()
    
    
    #oa_count_s = oa_count_s.fillna(0)
    
    return oa_count_s.sort_values(by=[sort], ascending=False).reset_index(drop=True)

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df['Repository'].value_counts()

In [ ]:
Michigan_df = df[df['institution_name'] == 'University of Michigan–Ann Arbor']
#ror_df = unnested_df[(unnested_df['ror'] == 'https://ror.org/0190ak572')]
Michigan_df['Repository'].value_counts()

In [ ]:
Michigan_df.groupby(['Repository', 'publication_year']).size().sort_values(ascending=False).head(15)


In [ ]:
encode_test = df[(df['Repository'] == 'ENCODE Datasets')]
len(encode_test) 

In [ ]:
# remove faculty opinions as it is not a dataset content
df = df.replace({'Faculty Opinions – Post-Publication Peer Review of the Biomedical Literature': 'Faculty Opinions Ltd'})

oa_df = df.copy()
oa_df = oa_df[oa_df['Repository'] != 'Faculty Opinions Ltd']

# Compare OpenAlex Dataset before and after it was created
OA2021df = oa_df[oa_df['publication_year'] <= 2021]
OA2022df = oa_df[oa_df['publication_year'] >= 2022]


In [ ]:
oa_df1 = counts('institution_name', 'is_oa', 'OA %', OA2021df).sort_values('institution_name', ascending=True)
oa_df2 = counts('institution_name', 'is_oa', 'OA %', OA2022df).sort_values('institution_name', ascending=True)
oa_df2 = oa_df2[oa_df2['institution_name'] != 'Michael J. Fox Foundation']

oa_df1 = oa_df1.rename(columns={'OA %': 'Before 2022 OA %'})
oa_df2 = oa_df2.rename(columns={'OA %': '2022 OA % Beyond'})

oa_df2 = oa_df2.reset_index(drop=True)
oa_df1 = pd.concat([oa_df1, oa_df2['2022 OA % Beyond']], axis=1)


# plotting sorted  by 2020 Diff 2018 on all there subjects (largest to smallest change)

oa_df_plot = oa_df1.plot(x="institution_name", y=['Before 2022 OA %', '2022 OA % Beyond'], kind="barh") 
oa_df_plot.set_title('Comparing OA % Across Institutions from before 2022 and 2022 Beyond')

oa_df_plot

In [ ]:
# Read in RADs Data

rads_df = pd.read_csv('RADS_Datasets_MN.csv')
rads_df.head()


In [ ]:
len(rads_df)

In [ ]:
# Read in RADs Data
rads_df = rads_df[['institution', 'id', 'DOI', 'group', 'publisher', 'publicationYear', 'resourceTypeGeneral', 'resourceType', 'schemaOrg', 'citeproc', 'source', 'state']]

rads_df = rads_df.rename(columns={'resourceType': 'subjectType', 'publisher': 'Repository', 'group': 'index location'})
rads_df = rads_df.rename(columns={'publisher': 'Repository'})	
#rads_df = rads_df.replace({'Harvard Dataverse': 'Harvard dataverse'})
rads_df = rads_df.replace({'Zenodo': 'Zenodo (CERN European Organization for Nuclear Research)', 'figshare': 'Figshare', 'Affiliation - Datacite': 'datacite', 'Affiliation - CrossRef': 'crossref'})


rads_df = rads_df.replace('dataset', 'Dataset')
rads_df.head()

In [ ]:
rads_df['index location'].value_counts()

In [ ]:
rads_df['resourceTypeGeneral'].value_counts()

In [ ]:
rads_df['subjectType'].value_counts().head(20)

In [ ]:
rads_df['citeproc'].value_counts().head(15)

In [ ]:
rads_df.head(15)

In [ ]:
# Make a copy of df and Rads to look at faculty opinions
OAFac_df = df.copy()
Rads_Fac_df = rads_df.copy()

# Look before and up to 2022 as that's when RADS ends and OA starts
OAFac_df = OAFac_df[OAFac_df['publication_year'] <= 2022]

# subset only to UMN datasets
OAFac_df = OAFac_df[OAFac_df['institution_name'] == 'University of Minnesota']


In [ ]:

MNFac_2022_repo_df = counts('Repository', 'is_oa', 'OA %', OAFac_df).reset_index(drop=True).sort_values(by=False, ascending=False).reset_index(drop=True)
#MNFac_repo_df = MNFac_repo_df.replace({'Faculty Opinions – Post-Publication Peer Review of the Biomedical Literature': 'Faculty Opinions Ltd'})
MNFac_2022_repo_df 

In [ ]:
RadsFacDF = RadsCounts('Repository', 'resourceTypeGeneral', Rads_Fac_df, 'Dataset')

#merged.plot(x="Repository", y=["RADS", "OpenAlex"], kind="barh", stacked=True).set_title('Rads Metadata Vs. OpenAlex Pre 2022 Metadata')
OARadsFacDf = pd.merge(RadsFacDF.head(15), MNFac_2022_repo_df, on='Repository', how='outer')
OARadsFacDf = OARadsFacDf.rename(columns={"Dataset": "RADS",  "Total": "OpenAlex"})

OARadsFacDf.plot(x="Repository", y=["RADS", "OpenAlex"], kind="barh").set_title('Rads Metadata Vs. OpenAlex Up To 2022 Repository Counts')

In [ ]:
# remove faculty opinions as it is not a dataset content
df = df[df['Repository'] != 'Faculty Opinions Ltd']
rads_df = rads_df[rads_df['Repository'] != 'Faculty Opinions Ltd']

# remove encode as it is not a dataset content
df = df[(df['Repository'] != 'ENCODE Datasets')]
rads_df = rads_df[rads_df['Repository'] != 'ENCODE Data Coordination Center']

# subset a new df to only UMN
MN_df = df[df['institution_name'] == 'University of Minnesota']

# subset two new dfs by year
#MN_2022df = MN_df[MN_df['publication_year'] >= 2022]
MN_2022_df = MN_df[MN_df['publication_year'] <= 2022]

In [ ]:
MN_repo_total = MN_df.groupby(['Repository'])['doi'].size().reset_index()
MN_repo_total = MN_repo_total.sort_values('doi',ascending=False)
MN_repo_total = MN_repo_total.rename(columns={ 'doi': 'DOI Count'})
MN_repo_total


In [ ]:
MN_repo_total.plot(x="Repository", y=['DOI Count'], kind="barh").set_title('OpenAlex Open Access Repository DOI Counts')

In [ ]:
MN_2022_repo_df = counts('Repository', 'is_oa', 'OA %', MN_2022_df).reset_index(drop=True).sort_values(by=False, ascending=False).reset_index(drop=True)
MN_2022_repo_df 

In [ ]:
RadsRepoDF = RadsCounts('Repository', 'resourceTypeGeneral', rads_df, 'Dataset')

#merged.plot(x="Repository", y=["RADS", "OpenAlex"], kind="barh", stacked=True).set_title('Rads Metadata Vs. OpenAlex Pre 2022 Metadata')
OARadsRepoDf = pd.merge(RadsRepoDF.head(15), MN_2022_repo_df, on='Repository', how='outer')
OARadsRepoDf = OARadsRepoDf.rename(columns={"Dataset": "RADS",  "Total": "OpenAlex"})

OARadsRepoDf.plot(x="Repository", y=["RADS", "OpenAlex"], kind="barh").set_title('Rads Vs. OpenAlex Repository Counts Up To 2022 Without Faculty Opinions')

In [ ]:
len(OAFac_df), len(Rads_Fac_df), len(MN_2022_df), len(rads_df)

In [ ]:
FacData = {'Repository': ['RADS', 'OpenAlex'], 'Includes Faculty Opinions': pd.Series([len(Rads_Fac_df), len(OAFac_df)]), 'No Faculty Opinions': pd.Series([len(rads_df), len(MN_2022_df)])}
FacOpinionsCounts_df = pd.DataFrame(data=FacData, index=[0, 1])
FacOpinionsCounts_df

In [ ]:
FacOpinionsCounts_df.plot(x="Repository", y=["Includes Faculty Opinions", "No Faculty Opinions"], kind="bar").set_title('Rads Vs. OpenAlex Faculty Opinions Dataset Count')

In [ ]:
# No FAculty Opinions subject type in RADS data
FacOpinionsSubjects_df = multiple_counts('Repository', 'field_name', 'is_oa', 'OA %', OAFac_df).reset_index(drop=True).sort_values(by=False, ascending=False).reset_index(drop=True)
FacOpinionsSubjects_df.head(20)

In [ ]:
Rads_Fac_df.head()

In [ ]:
#source = RadsCounts('Repository', 'source', rads_df, 'source')
Rads_Fac_df.groupby(['Repository'])['index location'].value_counts().reset_index().sort_values(by='count', ascending=False).head(20).reset_index(drop=True)

In [ ]:
# Make a subset of the data that is only faculty opinions
FacOnlyOA = OAFac_df[OAFac_df['Repository'] == 'Faculty Opinions Ltd']
FacOnlyRads =  Rads_Fac_df[Rads_Fac_df['Repository'] == 'Faculty Opinions Ltd']

In [ ]:
FacOnlyOAIndex = multiple_counts('Repository', 'index location', 'is_oa', 'OA %', FacOnlyOA).reset_index(drop=True).sort_values(by=False, ascending=False).reset_index(drop=True)

#FacOnlyOAIndex = FacOnlyOA.groupby(['Repository'])['index location'].value_counts().reset_index().sort_values(by='count', ascending=False).head(20).reset_index(drop=True)
FacOnlyOAIndex = FacOnlyOAIndex.rename(columns={"count": "OpenAlex"})

FacOnlyOAIndex


In [ ]:
df['index location'].value_counts()

In [ ]:
FacOnlyRadsIndex = FacOnlyRads.groupby(['Repository'])['index location'].value_counts().reset_index().sort_values(by='count', ascending=False).head(20).reset_index(drop=True)
FacOnlyRadsIndex = FacOnlyRadsIndex.rename(columns={"count": "RADS"})

FacOnlyRadsIndex

In [ ]:
OARadsIndexDf = pd.merge(FacOnlyOAIndex, FacOnlyRadsIndex, on='Repository', how='inner')
OARadsIndexDf = OARadsIndexDf.rename(columns={"count_x": "RADS",  "Total": "OpenAlex", "index location_x": "Index Location"})

OARadsIndexDf.plot(x="Index Location", y=["RADS", "OpenAlex"], kind="bar").set_title('OpenAlex Vs. RADS Faculty Opinion Index Location Counts')

In [ ]:
OA_FacYear_df = multiple_counts('Repository', 'publication_year', 'is_oa', 'OA %', FacOnlyOA).reset_index(drop=True).sort_values(by=False, ascending=False).reset_index(drop=True)
OA_FacYear_df = OA_FacYear_df.sort_values(by='Total', ascending=False).reset_index(drop=True)
OA_FacYear_df = OA_FacYear_df[['Repository', 'publication_year', 'Total']].sort_values(by='publication_year', ascending=False).reset_index(drop=True)
OA_FacYear_df

In [ ]:
Rads_FacYear_df = FacOnlyRads.groupby(['Repository'])['publicationYear'].value_counts().reset_index().sort_values(by='count', ascending=False).head(20).reset_index(drop=True)
Rads_FacYear_df = Rads_FacYear_df.rename(columns={"count": "RADS",  "Total": "OpenAlex", "publicationYear":"publication_year"})
Rads_FacYear_df = Rads_FacYear_df.sort_values(by='publication_year', ascending=False).reset_index(drop=True)


Rads_FacYear_df

In [ ]:
FacYear_df = pd.merge(OA_FacYear_df, Rads_FacYear_df, on='publication_year', how='outer')
FacYear_df = FacYear_df.rename(columns={"Repository_x": "Repository",  "Total": "OpenAlex"})

FacYear_df = FacYear_df[['Repository', 'publication_year', 'RADS', 'OpenAlex']]

FacYear_df['Repository'] = FacYear_df['Repository'].fillna('Faculty Opinions Ltd')

FacYear_df = FacYear_df.sort_values(by=['publication_year'], ascending=False)
FacYear_df

In [ ]:
FacYear_df.plot(x="publication_year", y=["RADS", "OpenAlex"], kind="bar").set_title('Faculty Opinions Publication Year Counts')


# Removing Faculty Opinions

In [ ]:
RadsYear_df = RadsCounts('publicationYear', 'resourceTypeGeneral', rads_df, 'publicationYear')
RadsYear_df = RadsYear_df.rename(columns={"Dataset": "RADS",  "publicationYear": "publication_year"})

RadsYear_df

#mn_year_df = multiple_counts('institution_name', 'publication_year', 'is_oa', 'OA %', MN_df).sort_values(by=['publication_year'], ascending=False)
#mn_year_df = mn_year_df.rename(columns={"Total": "OpenAlex"})

In [ ]:

OAYear_df = multiple_counts('institution_name', 'publication_year', 'is_oa', 'OA %', MN_df).sort_values(by=['publication_year'], ascending=False)
OAYear_df = OAYear_df.rename(columns={"Total": "OpenAlex"})
OAYear_df

In [ ]:

OARadsYearsDf = pd.merge(OAYear_df, RadsYear_df, on='publication_year', how='outer')
OARadsYearsDf = OARadsYearsDf.rename(columns={"publication_year": "Publication Year"})
OARadsYearsDf.plot(x="Publication Year", y=["RADS", "OpenAlex"], kind="bar").set_title('Rads Vs. OpenAlex Total Publication Counts by Year')

In [ ]:
RadsAllIndexDf = rads_df.groupby(['index location'])['index location'].value_counts().reset_index().sort_values(by='count', ascending=False).head(20).reset_index(drop=True)
RadsAllIndexDf = RadsAllIndexDf.rename(columns={"count": "RADS"})
RadsAllIndexDf = RadsAllIndexDf[['index location', 'RADS']]
RadsAllIndexDf

In [ ]:
OAAllIndexDf = MN_df.groupby(['index location'])['index location'].value_counts().reset_index().sort_values(by='count', ascending=False).head(20).reset_index(drop=True)
OAAllIndexDf = OAAllIndexDf.rename(columns={"count": "OpenAlex"})
OAAllIndexDf = OAAllIndexDf[['index location', 'OpenAlex']]
OAAllIndexDf

In [ ]:
OARadsIndexDf = pd.merge(RadsAllIndexDf, OAAllIndexDf, on='index location', how='outer')
OARadsIndexDf = OARadsIndexDf.rename(columns={"Dataset": "RADS",  "Total": "OpenAlex"})
OARadsIndexDf 

OARadsIndexDf.plot(x="index location", y=["RADS", "OpenAlex"], kind="barh").set_title('Rads Metadata Vs. OpenAlex Index Location Counts')


In [ ]:
rads_subject_df = rads_df[rads_df['subjectType'] != 'Dataset']


In [ ]:
RadsAllSubjectDf = rads_subject_df.groupby(['subjectType'])['subjectType'].value_counts().reset_index().sort_values(by='count', ascending=False).head(20).reset_index(drop=True)
RadsAllSubjectDf = RadsAllSubjectDf.rename(columns={"count": "RADS"})
RadsAllSubjectDf = RadsAllSubjectDf[['subjectType', 'RADS']]
RadsAllSubjectDf

In [ ]:
OAAllSubFieldDf = MN_df.groupby(['subfield_name'])['subfield_name'].value_counts().reset_index().sort_values(by='count', ascending=False).head(20).reset_index(drop=True)
OAAllSubFieldDf = OAAllSubFieldDf.rename(columns={"count": "OpenAlex", "subfield_name": 'subjectType'})
OAAllSubFieldDf = OAAllSubFieldDf[['subjectType', 'OpenAlex']]
OAAllSubFieldDf

In [ ]:
OARadsSubjectDf = pd.merge(RadsAllSubjectDf, OAAllSubFieldDf, on='subjectType', how='outer')
OARadsSubjectDf = OARadsSubjectDf.rename(columns={"Dataset": "RADS",  "Total": "OpenAlex"})
OARadsSubjectDf 

#OARadsSubjectDf.plot(x="subjectType", y=["RADS", "OpenAlex"], kind="barh").set_title('Rads Metadata Vs. OpenAlex Index Location Counts')


In [ ]:
MN_repo_df = counts('Repository', 'is_oa', 'OA %', MN_df).reset_index(drop=True).sort_values(by=False, ascending=False).reset_index(drop=True)
MN_repo_df = MN_repo_df[['Repository', False, True]]
MN_repo_df = MN_repo_df.rename(columns={False: 'Not OA', True: 'Is OA'})

MN_repo_df

In [ ]:
MN_repo_df.plot(x="Repository", y=['Is OA', 'Not OA'], kind="barh").set_title('OpenAlex Open Access by Repository')

In [ ]:
MN_field_df = counts('field_name', 'is_oa', 'OA %', MN_df).reset_index(drop=True).sort_values(by=False, ascending=False).reset_index(drop=True)
MN_field_df = MN_field_df[['field_name', False, True]]
MN_field_df = MN_field_df.rename(columns={False: 'Not OA', True: 'Is OA', 'field_name': 'Field Name'})

MN_field_df

In [ ]:
MN_field_df.plot(x="Field Name", y=['Is OA', 'Not OA'], kind="barh").set_title('OpenAlex Open Access by Field')

In [ ]:
MN_subfield_df = multiple_counts('subfield_name', 'Repository', 'is_oa', 'OA %', MN_df).sort_values(by=[False], ascending=False).reset_index(drop=True).head(25)
MN_subfield_df['Total'] = MN_subfield_df[True] + MN_subfield_df[False]

MN_subfield_df

In [ ]:
counts('domain_name', 'is_oa', 'OA %', MN_df).sort_values(by=[False], ascending=False).reset_index(drop=True).head(25)

In [ ]:
MN_departments_df = multiple_counts('raw_affiliation_strings', 'Repository', 'is_oa', 'OA %', MN_df).sort_values(by=[False], ascending=False).reset_index(drop=True).head(33)
MN_departments_df = MN_departments_df[['raw_affiliation_strings', False, True]]
MN_departments_df = MN_departments_df.rename(columns={False: 'Not OA', True: 'Is OA', 'raw_affiliation_strings': 'Department'}).head(25)
MN_departments_df['Department'] = MN_departments_df['Department'].str[0:50]
MN_departments_df

In [ ]:
MN_departments_df.plot(x="Department", y=['Is OA', 'Not OA'], kind="barh").set_title('OpenAlex Open Access by Department')

In [ ]:
MN_Year_df = counts('publication_year', 'is_oa', 'OA %', MN_df).sort_values(by='publication_year', ascending=False).reset_index(drop=True)
MN_Year_df = MN_Year_df[['publication_year', False, True]]
MN_Year_df = MN_Year_df.rename(columns={False: 'Not OA', True: 'Is OA', 'publication_year': 'Publication Year'})

MN_Year_df

In [ ]:
MN_Year_df.plot(x="Publication Year", y=['Is OA', 'Not OA'], kind="barh").set_title('OpenAlex Open Access by Publication Year')

In [ ]:
MNAuthor_df = counts('author_name','is_oa', 'OA %', MN_df).sort_values(by=[False], ascending=False).reset_index(drop=True)
MNAuthor_df = MNAuthor_df[['author_name', False, True]]
MNAuthor_df = MNAuthor_df.rename(columns={False: 'Not OA', True: 'Is OA',  'author_name': 'Author'})
MNAuthor_df['Author Not OA%'] = MNAuthor_df['Not OA'] / MNAuthor_df['Not OA'].sum()
MNAuthor_df = MNAuthor_df.head(10)
MNAuthor_df

In [ ]:
MNAuthor_plot = MNAuthor_df.plot(x="Author", y=['Is OA', 'Not OA'], kind="barh").set_title('OpenAlex Open Access by Author')

MNAuthor_plot

In [ ]:
MN_Author_Context_df = MN_df.groupby(['author_name', 'field_name', 'publication_year'])['is_oa'].value_counts().unstack('is_oa').sort_values(by=[False], ascending=False).reset_index()
MN_Author_Context_df = MN_Author_Context_df.fillna(0)
#MN_Author_Context_df['OA %'] = MN_Author_Context_df[True] / (MN_Author_Context_df[False] + MN_Author_Context_df[True])
#MN_Author_Context_df['Total'] = MN_Author_Context_df[True] + MN_Author_Context_df[False]

MN_Author_Context_df = MN_Author_Context_df.rename(columns={False: 'Not OA', True: 'Is OA', 'publication_year': 'Publication Year', 'author_name': 'Author', 'field_name': 'Field Name'})
MN_Author_Context_df['% Of Not Open Access'] = MN_Author_Context_df['Not OA'] / MN_Author_Context_df['Not OA'].sum()
MN_Author_Context_df = MN_Author_Context_df.head(10)
MN_Author_Context_df #, 'field_name'

In [ ]:
author_info_df = MN_df.groupby(['author_name'])['index location'].value_counts().reset_index().sort_values(by=['count'], ascending=False)
author_info_df = author_info_df.fillna(0).reset_index(drop=True)
author_info_df.head(25)

In [ ]:
author_infoRepo_df = MN_df.groupby(['author_name'])['Repository'].value_counts().reset_index().sort_values(by=['count'], ascending=False)
author_infoRepo_df = author_infoRepo_df.fillna(0).reset_index(drop=True)
author_infoRepo_df.head(25)

In [ ]:
author_infopos_df = MN_df.groupby(['author_name'])['author_position'].value_counts().reset_index().sort_values(by=['count'], ascending=False)
author_infopos_df = author_infopos_df.fillna(0).reset_index(drop=True)
author_infopos_df.head(25)

In [ ]:
author_infocollab_df = MN_df.groupby(['title'])['author_name'].value_counts().reset_index().sort_values(by=['count'], ascending=False)
author_infocollab_df = author_infocollab_df.fillna(0).reset_index(drop=True)
author_infocollab_df.head(25)

In [ ]:
author_inforetract_df = MN_df.groupby(['author_name'])['retracted'].value_counts().reset_index().sort_values(by=['count'], ascending=False)
author_inforetract_df = author_inforetract_df.fillna(0).reset_index(drop=True)
author_inforetract_df = author_inforetract_df[author_inforetract_df['retracted'] == True] 
author_inforetract_df.head(25)

In [ ]:
MN_Author_Index_df = MN_df.groupby(['author_name', 'index location'])['is_oa'].value_counts().unstack('is_oa').sort_values(by=[False], ascending=False).reset_index()
MN_Author_Index_df = MN_Author_Index_df.fillna(0)
#MN_Author_Context_df['OA %'] = MN_Author_Context_df[True] / (MN_Author_Context_df[False] + MN_Author_Context_df[True])
#MN_Author_Context_df['Total'] = MN_Author_Context_df[True] + MN_Author_Context_df[False]

MN_Author_Index_df = MN_Author_Index_df.rename(columns={False: 'Not OA', True: 'Is OA', 'publication_year': 'Publication Year', 'author_name': 'Author'})
MN_Author_Index_df['% Of Not Open Access'] = MN_Author_Index_df['Not OA'] / MN_Author_Index_df['Not OA'].sum()
MN_Author_Index_df = MN_Author_Index_df.head(25)
MN_Author_Index_df #, 'field_name'

In [ ]:
MN_Author_Repo_df = MN_df.groupby(['author_name', 'Repository'])['is_oa'].value_counts().unstack('is_oa').sort_values(by=[False], ascending=False).reset_index()
MN_Author_Repo_df = MN_Author_Repo_df.fillna(0)
#MN_Author_Context_df['OA %'] = MN_Author_Context_df[True] / (MN_Author_Context_df[False] + MN_Author_Context_df[True])
#MN_Author_Context_df['Total'] = MN_Author_Context_df[True] + MN_Author_Context_df[False]

MN_Author_Repo_df = MN_Author_Repo_df.rename(columns={False: 'Not OA', True: 'Is OA', 'publication_year': 'Publication Year', 'author_name': 'Author'})
MN_Author_Repo_df['% Of Not Open Access'] = MN_Author_Repo_df['Not OA'] / MN_Author_Repo_df['Not OA'].sum()
MN_Author_Repo_df = MN_Author_Repo_df.head(25)
MN_Author_Repo_df #, 'field_name'

In [ ]:
MN_Author_Orcid_df = MN_df.groupby(['author_name', 'orcid'])['is_oa'].value_counts().unstack('is_oa').sort_values(by=[False], ascending=False).reset_index()
MN_Author_Orcid_df = MN_Author_Orcid_df.fillna(0)
#MN_Author_Context_df['OA %'] = MN_Author_Context_df[True] / (MN_Author_Context_df[False] + MN_Author_Context_df[True])
#MN_Author_Context_df['Total'] = MN_Author_Context_df[True] + MN_Author_Context_df[False]

MN_Author_Orcid_df = MN_Author_Orcid_df.rename(columns={False: 'Not OA', True: 'Is OA', 'publication_year': 'Publication Year', 'author_name': 'Author'})
MN_Author_Orcid_df['% Of Not Open Access'] = MN_Author_Orcid_df['Not OA'] / MN_Author_Orcid_df['Not OA'].sum()
MN_Author_Orcid_df = MN_Author_Orcid_df.head(25)
MN_Author_Orcid_df #, 'field_name'

In [ ]:
MN_Author_afil_df = MN_df.groupby(['author_name', 'raw_affiliation_strings'])['is_oa'].value_counts().unstack('is_oa').sort_values(by=[False], ascending=False).reset_index()
MN_Author_afil_df = MN_Author_afil_df.fillna(0)
#MN_Author_Context_df['OA %'] = MN_Author_Context_df[True] / (MN_Author_Context_df[False] + MN_Author_Context_df[True])
#MN_Author_Context_df['Total'] = MN_Author_Context_df[True] + MN_Author_Context_df[False]

MN_Author_afil_df = MN_Author_afil_df.rename(columns={False: 'Not OA', True: 'Is OA', 'publication_year': 'Publication Year', 'author_name': 'Author'})
MN_Author_afil_df['% Of Not Open Access'] = MN_Author_afil_df['Not OA'] / MN_Author_afil_df['Not OA'].sum()
MN_Author_afil_df = MN_Author_afil_df.head(25)
MN_Author_afil_df #, 'field_name'

In [ ]:
MN_Author_position_df = MN_df.groupby(['author_name', 'author_position'])['is_oa'].value_counts().unstack('is_oa').sort_values(by=[False], ascending=False).reset_index()
MN_Author_position_df = MN_Author_position_df.fillna(0)
#MN_Author_Context_df['OA %'] = MN_Author_Context_df[True] / (MN_Author_Context_df[False] + MN_Author_Context_df[True])
#MN_Author_Context_df['Total'] = MN_Author_Context_df[True] + MN_Author_Context_df[False]

MN_Author_position_df = MN_Author_position_df.rename(columns={False: 'Not OA', True: 'Is OA', 'publication_year': 'Publication Year', 'author_name': 'Author'})
MN_Author_position_df['% Of Not Open Access'] = MN_Author_position_df['Not OA'] / MN_Author_position_df['Not OA'].sum()
MN_Author_position_df = MN_Author_position_df.head(25)
MN_Author_position_df #, 'field_name'

In [ ]:
MN_df

In [ ]:
remove_authors_df = MN_df[(MN_df.author_name != 'Phillip Portoghese') & (MN_df.author_name !='Ellen K. Longmire') & (MN_df.author_name !='Ankur Bordoloi')]
remove_authors_df = remove_authors_df.reset_index()

In [ ]:
new_field_df = counts('field_name','is_oa', 'OA %', remove_authors_df).sort_values(by=[False], ascending=False).reset_index(drop=True)
new_field_df = new_field_df[['field_name', False, True]]
new_field_df = new_field_df.rename(columns={False: 'Not OA', True: 'Is OA',  'field_name': 'Field'})
new_field_df

In [ ]:
new_field_df.plot(x="Field", y=['Is OA', 'Not OA'], kind="barh").set_title('OpenAlex Open Access by Author Without Outliers')

In [ ]:
new_subfield_df = counts('subfield_name','is_oa', 'OA %', remove_authors_df).sort_values(by=[True], ascending=False).reset_index(drop=True)
new_subfield_df = new_subfield_df.rename(columns={False: 'Not OA', True: 'Is OA',  'subfield_name': 'Subfield'})
#new_subfield_df = new_subfield_df[['subjectType', 'OpenAlex']]
new_subfield_df.head(20)
# remove_authors_df

In [ ]:
new_subfield_df.head(15).plot(x="Subfield", y=['Is OA', 'Not OA'], kind="barh").set_title('OpenAlex Open Access by SubField')

In [ ]:
new_subfield_df['Total'].sum()

In [ ]:
new_field_df['Is OA'].sum() + new_field_df['Not OA'].sum()

In [ ]:
new_field_df['Is OA'].sum()

In [ ]:
new_field_df['Not OA'].sum()

In [ ]:
MN_df

In [ ]:
mn_status_df = oa_status(MN_df, 'publication_year', 'oa_status', 'publication_year')
mn_status_df

In [ ]:
mn_status_plot = mn_status_df.plot(x="publication_year", y=["bronze", "closed", "gold", "green", "hybrid"], kind="bar")
mn_status_plot

## Plant Sciences    

In [ ]:
plt_df = MN_df.groupby(['subfield_name'])['subfield_name'].value_counts().reset_index().sort_values(by='count', ascending=False).head(20).reset_index(drop=True)
plt_df = plt_df.rename(columns={"count": "OpenAlex", "subfield_name": 'subjectType'})
plt_df = plt_df[['subjectType', 'OpenAlex']]
plt_df

In [ ]:
MN_df['field_name'].unique()
# Agricultural and Biological Sciences
## Environmental Science, Biochemistry, Genetics and Molecular Biology


In [ ]:
field_txt = MN_df
# only genetics subfield needed from biology one
# only Global and Planetary Change from es maybe Management, Monitoring, Policy and Law
field_txt = field_txt[(field_txt['field_name'] == 'Agricultural and Biological Sciences') | (field_txt['field_name'] == 'Environmental Science') | (field_txt['field_name'] == 'Biochemistry, Genetics and Molecular Biology')].reset_index(drop=True)

field_txt.to_csv('field_text.csv')

### subfield

In [ ]:
MN_df['subfield_name'].unique()

# Plant Science, General Agricultural and Biological Sciences, Ecology, Evolution, Behavior and Systematics
## soil sience, Genetics, Food Science, Sustainability and the Environment, Nature and Landscape Conservation
### Developmental Biology, Cell Biology, Molecular Biology, Ecological Modeling
# ? nan

In [ ]:
subfields_txt = MN_df
subfields_txt = subfields_txt[(subfields_txt['subfield_name'] == 'Plant Science') | (subfields_txt['subfield_name'] == 'General Agricultural and Biological Sciences') 
                              | (subfields_txt['subfield_name'] == 'Genetics') | (subfields_txt['subfield_name'] == 'Molecular Biology') | (subfields_txt['subfield_name'] == 'Renewable Energy, Sustainability and the Environment') | (subfields_txt['subfield_name'] == 'Global and Planetary Change from es maybe Management') ].reset_index(drop=True)
subfields_txt
subfields_txt.to_csv('subfield_text.csv')

In [ ]:
MN_df['topic_name'].unique()

# Plant and animal studies, Botany and Plant Ecology Studies,  Genetic Mapping and Diversity in Plants and Animals, Mycorrhizal Fungi and Plant Interactions, Agricultural risk and resilience, Microbial Metabolic Engineering and Bioproduction, Microtubule and mitosis dynamics, Wheat and Barley Genetics and Pathology, Agriculture and Rural Development Research
## Soil Carbon and Nitrogen Dynamics, Molecular Biology Techniques and Applications, Insect Pest Control Strategies, Nutrition, Genetics, and Disease, Economics of Agriculture and Food Markets, Genetics, Bioinformatics, and Biomedical Research, Biochemical and Molecular Research, Algal biology and biofuel production
### Membrane-based Ion Separation Techniques, Microfluidic and Bio-sensing Technologies, Sustainable Development and Environmental Policy, CAR-T cell therapy research, Metabolism and Genetic Disorders, Enzyme Catalysis and Immobilization, Biotin and Related Studies, Water Quality and Resources Studies, Mediterranean and Iberian flora and fauna, Agricultural Economics and Policy

In [ ]:
topic_txt = MN_df
topic_txt = topic_txt[(topic_txt['topic_name'] == 'Plant and animal studies') | (topic_txt['topic_name'] == 'Botany and Plant Ecology Studies') |(topic_txt['topic_name'] == 'Genetic Mapping and Diversity in Plants and Animals') |(topic_txt['topic_name'] == 'Microtubule and mitosis dynamics') |(topic_txt['topic_name'] == 'Wheat and Barley Genetics and Pathology') |(topic_txt['topic_name'] == 'Agriculture and Rural Development Research')
                      |(topic_txt['topic_name'] == 'Molecular Biology Techniques and Applications') |(topic_txt['topic_name'] == 'Insect Pest Control') |(topic_txt['topic_name'] == 'Algal biology and biofuel production') | (topic_txt['topic_name'] == 'Mediterranean and Iberian flora and fauna') ].reset_index(drop=True)
topic_txt


In [ ]:
###  ,  Agricultural Economics and Policy

In [ ]:
citation_df = df[['id',
 'doi',
 'type',
 'funding_type',
 'title',
 'publication_year',
 'publication_date',
 'language',
 'author_id',
 'author_name',
 'author_position',
 'orcid',
 'has_fulltext',
 'cited_by_count',
 'apc_list',
 'apc_paid',
 'citation_normalized_value',
 'is_in_top_1_percent',
 'is_in_top_10_percent',
 'min',
 'max',
 'topic_id',
 'topic_name',
 'subfield_id',
 'subfield_name',
 'field_id',
 'field_name',
 'domain_id',
 'domain_name',
 'retracted',
 'index location',
 'Repository',
 'is_oa',
 'oa_status',
 'license',
 'institution_id',
 'institution_name',
 'raw_affiliation_strings',
 'country_code',
 'ror',
 'lineage']]


In [ ]:
df.groupby(['institution_name'])['apc_list'].count().reset_index()

In [ ]:
df.groupby(['field_name'])['apc_list'].count().reset_index()

In [ ]:
df['fwci'].notnull().any()

In [ ]:
citation_df.head()

In [ ]:
institution_citation_count = df.groupby(['institution_name'])['cited_by_count'].sum().reset_index().sort_values(by='cited_by_count', ascending=False).reset_index(drop=True)
institution_citation_count['Citation %'] = institution_citation_count['cited_by_count'] / institution_citation_count['cited_by_count'].sum()
institution_citation_count

In [ ]:
institution_citation_normalized = df.groupby(['institution_name'])['citation_normalized_value'].mean().reset_index().sort_values(by='citation_normalized_value', ascending=False).reset_index(drop=True)
institution_citation_normalized

In [ ]:
institution_citation_top1 = df.groupby(['institution_name'])['is_in_top_1_percent'].mean().reset_index().sort_values(by='is_in_top_1_percent', ascending=False).reset_index(drop=True)
institution_citation_top1

In [ ]:
institution_citation_top10 = df.groupby(['institution_name'])['is_in_top_10_percent'].mean().reset_index().sort_values(by='is_in_top_10_percent', ascending=False).reset_index(drop=True)
institution_citation_top10

In [ ]:
institution_citation_min = df.groupby(['institution_name'])['min'].mean().reset_index().sort_values(by='min', ascending=False).reset_index(drop=True)
institution_citation_min = institution_citation_min.rename(columns={"min": "Mean Min"})
institution_citation_min


In [ ]:
institution_citation_max = df.groupby(['institution_name'])['max'].mean().reset_index().sort_values(by='max', ascending=False).reset_index(drop=True)
institution_citation_max = institution_citation_max.rename(columns={"max": "Mean Max"})
institution_citation_max

In [ ]:
institution_citation_fulltxt = df.groupby(['institution_name'])['has_fulltext'].mean().reset_index().sort_values(by='has_fulltext', ascending=False).reset_index(drop=True)
institution_citation_fulltxt

In [ ]:
institution_citations = pd.merge(institution_citation_count, institution_citation_normalized, on='institution_name')
institution_citations = pd.merge(institution_citations, institution_citation_top1, on='institution_name')
institution_citations = pd.merge(institution_citations, institution_citation_top10, on='institution_name')
institution_citations = pd.merge(institution_citations, institution_citation_min, on='institution_name')
institution_citations = pd.merge(institution_citations, institution_citation_max, on='institution_name')
institution_citations = pd.merge(institution_citations, institution_citation_fulltxt, on='institution_name')
institution_citations.sort_values(by='is_in_top_1_percent', ascending=False).reset_index(drop=True)

In [ ]:
field_citation_count = df.groupby(['field_name'])['cited_by_count'].sum().reset_index().sort_values(by='cited_by_count', ascending=False).reset_index(drop=True)
field_citation_count['citation %'] = field_citation_count['cited_by_count'] / field_citation_count['cited_by_count'].sum()
field_citation_count


In [ ]:
field_citation_normalized = df.groupby(['field_name'])['citation_normalized_value'].mean().reset_index().sort_values(by='citation_normalized_value', ascending=False).reset_index(drop=True)
field_citation_normalized

In [ ]:
field_citation_top1 = df.groupby(['field_name'])['is_in_top_1_percent'].mean().reset_index().sort_values(by='is_in_top_1_percent', ascending=False).reset_index(drop=True)
field_citation_top1

In [ ]:
t = df.groupby(['field_name'])['is_in_top_1_percent'].value_counts().unstack('is_in_top_1_percent').reset_index()
t = t.fillna(0)
t['%'] = t[True] / (t[False] + t[True])
t['Total'] = t[True] + t[False]
t.sort_values(by='%', ascending=False).reset_index(drop=True)

In [ ]:
field_citations_top10 = df.groupby(['field_name'])['is_in_top_10_percent'].mean().reset_index().sort_values(by='is_in_top_10_percent', ascending=False).reset_index(drop=True)
field_citations_top10

In [ ]:
field_citation_min = df.groupby(['field_name'])['min'].mean().reset_index().sort_values(by='min', ascending=False).reset_index(drop=True)
field_citation_min = field_citation_min.rename(columns={"min": "Mean Min"})
field_citation_min

In [ ]:
field_citation_max = df.groupby(['field_name'])['max'].mean().reset_index().sort_values(by='max', ascending=False).reset_index(drop=True)
field_citation_max = field_citation_max.rename(columns={"max": "Mean Max"})
field_citation_max

In [ ]:
field_citation_fulltxt = df.groupby(['field_name'])['has_fulltext'].mean().reset_index().sort_values(by='has_fulltext', ascending=False).reset_index(drop=True)
field_citation_fulltxt

In [ ]:
field_citations = pd.merge(field_citation_count, field_citation_normalized, on='field_name')
field_citations = pd.merge(field_citations, field_citation_top1, on='field_name')
field_citations = pd.merge(field_citations, field_citations_top10, on='field_name')
field_citations = pd.merge(field_citations, field_citation_min, on='field_name')
field_citations = pd.merge(field_citations, field_citation_max, on='field_name')
field_citations = pd.merge(field_citations, field_citation_fulltxt, on='field_name')
field_citations.sort_values(by='is_in_top_1_percent', ascending=False).reset_index(drop=True)

In [ ]:
len(df[df['cited_by_count'] >= 100 ].reset_index(drop=True).sort_values(by=['cited_by_count'], ascending=True).reset_index(drop=True))


In [ ]:
df.groupby(['author_name', 'author_position'])['is_oa'].value_counts().unstack('is_oa').sort_values(by=[False], ascending=False).reset_index()


In [ ]:
df.head()